# Demo de inferencia (Fase 6): pipeline YOLO + verificador

Ejecuta el modelo final sobre un video corto no visto en el entrenamiento y aplica el verificador determinista. Exporta un video anotado con bboxes coloreados por estado y tasa de cumplimiento por fotograma.

> **Notebook delgado**: solo orquesta. La lógica vive en `src/` del repositorio.

In [ ]:
# Setup en Kaggle: clona el repo del proyecto e instala dependencias.
# El código pesado vive en `src/`; este notebook solo orquesta.
import os, subprocess, sys

REPO_URL = "https://github.com/<usuario>/<repo>.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

In [ ]:
# Inferencia frame a frame sobre un video corto + verificador determinista.
# El video DEBE ser no visto durante el entrenamiento.
import cv2
from pathlib import Path
from ultralytics import YOLO
from src.compliance.verifier import Detection, verify_frame
from src.compliance.renderer import annotate_frame

# TODO: descargar el best.pt del release de Git y subirlo como Kaggle Dataset,
# o subir el archivo manualmente al notebook antes de ejecutar esta celda.
WEIGHTS = "/kaggle/input/epp-yolo-weights/best.pt"
INPUT_VIDEO = "/kaggle/input/epp-demo-videos/demo.mp4"
OUTPUT_VIDEO = "/kaggle/working/demo_annotated.mp4"

model = YOLO(WEIGHTS)

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w, h))

while True:
    ok, frame = cap.read()
    if not ok:
        break
    results = model.predict(frame, verbose=False)[0]
    dets = []
    for box, cls, conf in zip(results.boxes.xyxy.cpu().numpy(),
                              results.boxes.cls.cpu().numpy(),
                              results.boxes.conf.cpu().numpy()):
        dets.append(Detection(class_id=int(cls), box=tuple(box.tolist()), score=float(conf)))
    report = verify_frame(dets)
    annotated = annotate_frame(frame, report)
    writer.write(annotated)

cap.release()
writer.release()
print(f"Video anotado escrito en {OUTPUT_VIDEO}")